In [ ]:
# Lab type: prompt
# Course: AI-Assisted Data Science
# Lesson: Prompt Engineering for Data Tasks
# Task: Practice writing schema+goal+constraints prompts, critique AI-generated code,
#       and compose narrow debugging prompts using a real employee dataset

# Lab: Prompt Engineering for Data Tasks

This lab is about writing prompts, not reading them.

You'll work through four exercises using the `employee_records` dataset:

1. **Weak vs. strong prompts** — identify what a vague prompt is missing
2. **Write a prompt** — build a full schema+goal+constraints prompt from scratch
3. **Critique an AI pipeline** — write the follow-up prompt that surfaces the flaws
4. **Narrow debugging** — turn a surprising result into a precise debugging prompt

Complete the setup section first. The inspection output is the raw material you'd paste
into every prompt you write today.

## Setup: load and inspect the dataset

In [ ]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/SophiArch/notebooks/main/datasets/employee_records.csv"
df = pd.read_csv(url)

print(df.shape)
print()
print(df.dtypes)

In [ ]:
df.describe(include="all").round(2)

In [ ]:
df.head(3)

In [ ]:
print("Missing values:")
print(df.isna().sum())
print()
print("Salary percentiles:")
print(df["salary"].quantile([0.25, 0.5, 0.75, 0.95, 0.99]).round(0))

## Exercise 1 — Weak vs. strong prompts

A colleague asks an AI tool to help prepare features for a salary prediction model.
They send this prompt:

> **Weak prompt:**
> "Write Python code to prepare the data for a salary prediction model."

Contrast it with this version:

> **Strong prompt:**
> "I have a DataFrame `df` with these columns and dtypes:
> - `employee_id`: int64 — unique identifier, should not be a feature
> - `department`: object — nominal categorical
> - `role`: object — nominal categorical (high cardinality, ~40 unique values)
> - `salary`: float64 — **target variable**; ~2.5% nulls; heavy right skew, 99th percentile ≈ $197k, max ≈ $310k
> - `years_exp`: int64 — no nulls
> - `performance`: object — ordinal: 'below_expectations' < 'meets_expectations' < 'exceeds_expectations'; ~3% nulls
> - `region`: object — nominal categorical
>
> Shape: (1201, 7).
>
> **Task:** Prepare `X` and `y` for a salary regression model.
>
> **Constraints:**
> - Drop `employee_id` — it is an identifier, not a feature.
> - `salary` is the target; cap it at the 99th percentile of the **training set** before scaling.
> - Impute `salary` nulls with the training-set median (median is more robust than mean given the skew).
> - Encode `performance` as an ordered ordinal (below=0, meets=1, exceeds=2).
> - One-hot encode `department` and `region`; target-encode `role` (high cardinality).
> - Fit all transformers on the training set only; apply to test with transform().
> - Use `random_state=42` and `test_size=0.2`."

The cell below re-runs the schema inspection you did in setup so you can refer to it here.

In [ ]:
print("Columns and dtypes:")
print(df.dtypes)
print()
print("Nulls:")
print(df.isna().sum())
print()
print("Salary distribution:")
print(df["salary"].describe().round(0))
print()
print("Performance values:")
print(df["performance"].value_counts(dropna=False))

**Your turn.** The weak prompt is missing several things that the strong prompt provides.
List at least four specific pieces of information the AI cannot infer from the weak prompt.

*(Write your answer here.)*

- 
- 
- 
- 

<details>
<summary>🔑 Reveal answer — Exercise 1</summary>

At least four things the AI cannot infer from "Write Python code to prepare the data for a salary prediction model":

1. **Which column is the target:** Without being told, the AI might include `salary` as a feature or pick a different column entirely.
2. **Which columns to exclude:** `employee_id` is an identifier — a model that treats it as a numeric feature may memorise id-to-label mappings that don't generalise.
3. **Encoding strategy per column:** `performance` is ordinal; `department` and `region` are nominal; `role` is high-cardinality nominal. The AI has no way to infer these distinctions or the correct ordering of performance levels.
4. **Imputation strategy and outlier handling:** The weak prompt gives no hint that `salary` is right-skewed (median imputation is more robust than mean), or that values above the 99th percentile should be capped.
5. **Train/test leakage constraint:** The AI may fit transformers on the full dataset. Specifying "fit on training data only; apply to test with `transform()`" prevents this silently bad default.

</details>

## Exercise 2 — Write a prompt

Write a complete schema+goal+constraints prompt for the following task:

> Impute missing `salary` values using the median, fitted on training data only.
> Cap `salary` at the 99th percentile of the training set before scaling.
> Do not include `employee_id` as a feature.

Use the template below. Fill in every `[...]` using the output from the setup cells.
Do not leave any blank — a complete prompt gives the AI everything it needs with nothing left to assume.

*(Complete the prompt template below.)*

> I have a DataFrame `df` with shape `[...]` and the following columns:
>
> - `employee_id`: `[dtype]` — `[role: identifier / feature / target]`
> - `department`: `[dtype]` — `[description]`
> - `role`: `[dtype]` — `[description]`
> - `salary`: `[dtype]` — `[description including null rate and skew facts]`
> - `years_exp`: `[dtype]` — `[description]`
> - `performance`: `[dtype]` — `[description including ordering and null rate]`
> - `region`: `[dtype]` — `[description]`
>
> **Task:** `[State the specific modelling goal in one sentence.]`
>
> **Constraints:**
> - `[Constraint about employee_id]`
> - `[Constraint about imputing salary — method and which split it must be fitted on]`
> - `[Constraint about the 99th-percentile cap — when to compute it]`
> - `[Constraint about scaling — when to fit, when to only transform]`
> - `[Any additional constraint you consider important given the data]`

<details>
<summary>🔑 Model prompt — Exercise 2</summary>

**Example strong prompt:**

> I have a DataFrame `df` with shape `(1201, 7)` and the following columns:
>
> - `employee_id`: `int64` — unique identifier; **drop before modelling, not a feature**
> - `department`: `object` — nominal categorical; one-hot encode
> - `role`: `object` — nominal categorical, ~40 unique values (high cardinality); target-encode
> - `salary`: `float64` — **target variable**; ~2.5% nulls; heavy right skew (99th percentile ≈ $197k, max ≈ $310k); impute nulls with training-set median; cap at the 99th percentile of the training set using `np.clip`
> - `years_exp`: `int64` — numeric feature; no nulls
> - `performance`: `object` — ordinal: `'below_expectations'` < `'meets_expectations'` < `'exceeds_expectations'`; ~3% nulls; encode as ordered ordinal (0, 1, 2); impute nulls with training-set most-frequent value
> - `region`: `object` — nominal categorical; one-hot encode
>
> **Task:** Prepare `X` and `y` for a salary regression model using `train_test_split(test_size=0.2, random_state=42)`.
>
> **Constraints:**
> - Drop `employee_id` — identifier, not a feature.
> - Impute `salary` nulls using the **training-set median** (median is preferred given right skew).
> - Cap `salary` at the **99th percentile computed on the training set only** before scaling.
> - Fit all transformers on training data; apply to test with `transform()` only — no `fit_transform` on test or full data.
> - Encode `performance` with explicit ordinal order: `['below_expectations', 'meets_expectations', 'exceeds_expectations']`.

**Why it's strong:** Every ambiguity is resolved — target column, identifier to drop, encoding strategy per column, imputation method with its rationale, leakage constraint, and outlier handling. The AI has nothing left to assume.

</details>

## Exercise 3 — Critique an AI-generated pipeline

A teammate pasted your prompt into an AI tool and received the pipeline below.
Read it — **do not run it** — then write the critique prompt you would send back.

A good critique prompt asks the AI to explain its reasoning on specific lines,
point out the assumptions it made, and describe what could go wrong.

In [ ]:
# --- AI-GENERATED CODE: read, do not run ---
#
# import pandas as pd
# import numpy as np
# from sklearn.impute import SimpleImputer
# from sklearn.preprocessing import StandardScaler
# from sklearn.model_selection import train_test_split
#
# # Drop identifier
# X = df.drop(columns=["employee_id", "salary"])
# y = df["salary"]
#
# # Impute salary nulls with mean
# salary_imputer = SimpleImputer(strategy="mean")
# y_imputed = salary_imputer.fit_transform(y.values.reshape(-1, 1)).ravel()
#
# # Cap salary at 99th percentile
# cap = np.percentile(y_imputed, 99)
# y_capped = np.clip(y_imputed, None, cap)
#
# # Scale features
# numeric_cols = X.select_dtypes(include="number").columns
# scaler = StandardScaler()
# X[numeric_cols] = scaler.fit_transform(X[numeric_cols])
#
# # Split
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y_capped, test_size=0.2, random_state=42
# )

print("Code read. Continue to the critique exercise.")

**Part A.** Write the critique prompt you would send to the AI.
It should ask the AI to explain specific decisions and flag potential problems.
Reference the exact lines or variable names you are asking about.

*(Write your critique prompt here.)*

> "The code you wrote has a few decisions I want to understand. For each question below,
> explain your reasoning and describe what could go wrong:
>
> 1. `[Ask about the imputation strategy choice on the salary column.]`
> 2. `[Ask about when the 99th-percentile cap is computed relative to the train/test split.]`
> 3. `[Ask about when the scaler is fitted relative to the train/test split.]`
> 4. `[Ask about anything else in the pipeline you want the AI to justify.]"`

<details>
<summary>🔑 Model critique prompt — Exercise 3 Part A</summary>

**Example strong critique prompt:**

> "The code you produced has a few decisions I want to understand before I use it. For each question, explain your reasoning and describe what could go wrong:
>
> 1. On the line `salary_imputer = SimpleImputer(strategy="mean")`: you chose `mean` for the `salary` column. The column has a heavy right skew with outliers reaching $310k. How does using the mean rather than the median affect the imputed values for missing rows, and when would this matter for model accuracy?
>
> 2. The line `cap = np.percentile(y_imputed, 99)` runs on the full dataset before `train_test_split`. This means the 99th-percentile threshold is computed using test-set values. At what point in the pipeline should this cap be computed, and what information does computing it on the full dataset leak?
>
> 3. `scaler.fit_transform(X[numeric_cols])` is called on the full `X` before the split. What statistics does the scaler learn here, and why does fitting on both training and test rows make the test-set evaluation artificially optimistic?
>
> 4. After the split, `X_test` was already scaled as part of the full dataset. If this pipeline were deployed, how would you scale new inference-time rows — and is there a clean `fit`/`transform` separation in the current code?"

</details>

**Part B.** Without running the code, identify the concrete issues in the pipeline.
For each issue, state what the AI did, what it should have done, and why it matters.

*(Write your answer here.)*

| Issue | What the AI did | What it should have done | Why it matters |
|-------|----------------|--------------------------|----------------|
| Imputation strategy | | | |
| Cap computation timing | | | |
| Scaler fitting timing | | | |

<details>
<summary>🔑 Reveal answer — Exercise 3 Part B</summary>

| Issue | What the AI did | What it should have done | Why it matters |
|-------|----------------|--------------------------|----------------|
| **Imputation strategy** | Used `SimpleImputer(strategy="mean")` on `salary` | Use `strategy="median"` | With heavy right skew and outliers near $310k, the mean is pulled upward; imputed rows get values too high for most missing employees |
| **Cap computation timing** | Computed the 99th-percentile cap on the full `y_imputed` before splitting | Compute the cap on `y_train` only, after splitting | The cap threshold includes test-set salary values — this leaks future distribution information into the transformation |
| **Scaler fitting timing** | Called `fit_transform(X[numeric_cols])` on the full dataset before splitting | Split first; `fit_transform` on `X_train` only; `transform` on `X_test` | The scaler learns test-set statistics; test metrics appear better than they would be in production, where new data is always unseen |

</details>

## Exercise 4 — Narrow debugging

Run the cell below. It groups mean salary by region — but the output is missing
some regions you'd expect to see, and the counts are lower than expected.

Inspect the output carefully before writing your debugging prompt.

In [ ]:
result = (
    df[df["performance"] == "exceeds_expectations"]
    .groupby("region")["salary"]
    .agg(["mean", "count"])
    .round(0)
)
print(result)

In [ ]:
print("All regions in df:")
print(sorted(df["region"].unique()))
print()
print("Total rows with performance == 'exceeds_expectations':")
print((df["performance"] == "exceeds_expectations").sum())
print()
print("Rows where performance is null:")
print(df["performance"].isna().sum())
print()
print("Rows where salary is null (in exceeds_expectations subset):")
exceeds = df[df["performance"] == "exceeds_expectations"]
print(exceeds["salary"].isna().sum())

**Your turn.** Write the narrow debugging prompt you would send to the AI.

A narrow debugging prompt has three parts:
1. What you expected to see
2. What you actually got (with the specific discrepancy)
3. What you have already checked

*(Write your debugging prompt here.)*

> "I ran the following code on a DataFrame with shape `[...]`:
>
> ```python
> [paste the groupby code here]
> ```
>
> **Expected:** `[Describe what you expected the output to contain.]`
>
> **Actual:** `[Describe exactly what was missing or surprising.]`
>
> **Already checked:**
> - `[State one thing you have already verified about the data.]`
> - `[State another thing you have already verified.]`
>
> What is causing this and how should I fix it?"

<details>
<summary>🔑 Model debugging prompt — Exercise 4</summary>

**Example strong narrow debugging prompt:**

> "I ran the following code on a DataFrame `df` with shape `(1201, 7)`:
>
> ```python
> result = (
>     df[df["performance"] == "exceeds_expectations"]
>     .groupby("region")["salary"]
>     .agg(["mean", "count"])
>     .round(0)
> )
> print(result)
> ```
>
> **Expected:** A row for each of the 5 regions in `df` (`['Central', 'East', 'North', 'South', 'West']`). The total `count` across all rows should match `(df["performance"] == "exceeds_expectations").sum()`.
>
> **Actual:** Fewer than 5 regions appear in the output, and the total count is lower than expected. Some regions are missing entirely.
>
> **Already checked:**
> - All 5 region values exist in `df["region"]` — `df["region"].unique()` confirms no typos.
> - The performance filter is working: `(df["performance"] == "exceeds_expectations").sum()` returns the expected count.
> - There are N rows where `df["performance"]` is `NaN` (those are correctly excluded by the `==` filter).
>
> What is causing rows to be silently dropped from the `groupby`, and how should I fix it?"

**Why it's strong:** It distinguishes what was expected from what was observed, references the exact aggregation code, and documents three prior checks so the AI doesn't re-suggest things already ruled out. The root cause here is `groupby` dropping `NaN` region keys by default — fix with `groupby("region", dropna=False)`.

</details>

## Reflection

Summarise what you have practised in this lab.
Write 3–5 bullet points describing what makes a good data prompt
and what you will do differently when prompting AI tools in future.

*(Write your reflection here.)*

- 
- 
- 